# Test PII-Tracer with a synthetic email
Run the cells in order. Edit `synthetic_email` to try different fictional examples.
The notebook reads the repository's local `.env`; it never prints your API key.
It sends text to the hosted `pii-tracer` model through `/v1/pii`.

Use synthetic data for these tests. Detection can miss PII or mask harmless text;
this is an inspection notebook, not proof that a document is safe to publish.
Inputs above 4096 tokens are rejected and must be split explicitly.
Sensitivity is a separate conversation-level score. Decoded spans are masked even when this score is low.

In [ ]:
%pip install -q python-dotenv==1.2.4

In [ ]:
from pathlib import Path
import json
from urllib.request import Request, build_opener, HTTPRedirectHandler
from dotenv import dotenv_values

root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".env").is_file()), None)
assert root is not None, "Start Jupyter inside the project folder."
settings = dotenv_values(root / ".env", interpolate=False)
base_url = settings["UCLOUD_PUBLIC_BASE_URL"].rstrip("/")
if not base_url.startswith("https://"): base_url = "https://" + base_url
if not base_url.endswith("/v1"): base_url += "/v1"
api_key = settings["VLLM_API_KEY"]
assert api_key, "Set VLLM_API_KEY in the local .env."
model = "pii-tracer"

class NoRedirects(HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        return None

def call(route, payload=None):
    body = json.dumps(payload).encode("utf-8") if payload is not None else None
    request = Request(base_url + route, data=body, headers={
        "Authorization": "Bearer " + api_key, "Content-Type": "application/json"})
    with build_opener(NoRedirects()).open(request, timeout=120) as response:
        return json.loads(response.read())

assert model in [m["id"] for m in call("/models")["data"]], "The expected model is not hosted."
print("Connected to", model)

## Vælg eller redigér en syntetisk mail
Der er to danske eksempler: en mødeaftale med kontaktoplysninger og en mail til IT-support med en opdigtet adgangskode.
Det engelske eksempel er også bevaret. Vælg `selected_example` i cellen nedenfor, og kør de efterfølgende celler igen.
Du kan også redigere mailteksterne direkte. Alle oplysninger er syntetiske; eksemplerne viser, hvad modellen finder, uden at forudsætte at den finder alt.

In [ ]:
# Alle navne, adresser, numre og adgangskoder her er opdigtede testdata.
email_examples = {
    'møde': """Fra: Anna Lund <anna.lund@example.org>
Til: Mikkel Sørensen <mikkel.sorensen@example.net>
Emne: Aftale om møde og kontaktoplysninger

Hej Mikkel,

Tak for din besked. Jeg hedder Anna Lund og vil gerne aftale et møde
på torsdag kl. 10. Du kan kontakte mig på anna.lund@example.org
eller telefon +45 99 88 77 66.

Min adresse er Eksempelvej 123, 1234 Eksempelby.
Send gerne en bekræftelse på mail, når tidspunktet er på plads.

Venlig hilsen
Anna Lund
""",
    'it-support': """Fra: Emil Hansen <emil.hansen@example.org>
Til: IT-support <support@example.net>
Emne: Hjælp til adgang til min konto

Hej IT-support,

Jeg hedder Emil Hansen og kan ikke logge ind på min konto.
Mit brugernavn er emil.hansen@example.org, og den midlertidige
adgangskode er EksempelKode!2026.

Mit kundenummer er TEST-KUNDE-0042. Ring gerne til mig på
+45 88 77 66 55, hvis I har brug for flere oplysninger.

Jeg vil gerne bede om et nyt link til nulstilling af adgangskoden.

Med venlig hilsen
Emil Hansen
""",
    'english': """From: Anna Lund <anna.lund@example.org>
To: Daniel Whitfield <daniel.whitfield@example.net>
Subject: Meeting and account access

Hi Daniel,

My name is Anna Lund. Please contact me at anna.lund@example.org
or call +1 202-555-0147. My mailing address is 123 Example Street,
Springfield, IL 62704.

For this fictional test, the temporary password is MapleRiver!2026.
Please keep this information private.

Best regards,
Anna Lund
""",
}

# Vælg "møde", "it-support" eller "english", og kør resten af cellerne igen.
selected_example = "møde"
synthetic_email = email_examples[selected_example]
print(synthetic_email)


In [ ]:
response = call("/pii", {"model": model, "sequences": [synthetic_email]})
assert response["model"] == model
result = response["results"][0]
assert all(synthetic_email[s["start"]:s["end"]] == s["text"] for s in result["spans"])
print("Tokens:", result["token_count"])
print("Sensitivity:", round(result["sensitivity"], 4))
print("Detected spans:", len(result["spans"]))

In [ ]:
from IPython.display import display, HTML
from html import escape

rows = "".join(
    "<tr>" + "".join("<td>" + escape(str(value)) + "</td>" for value in
        (span["pii_type"], span["text"], span["start"], span["end"], round(span["score"], 4))) + "</tr>"
    for span in result["spans"])
display(HTML("<table><thead><tr><th>Category</th><th>Detected text</th><th>Start</th><th>End</th><th>Score</th></tr></thead><tbody>" + rows + "</tbody></table>"))
print("Masked email:\n")
print(result["masked_text"])

## Optional: inspect raw scoring alignment
`/v1/scoring` returns 37 token-label logits and a separate sensitivity logit.
The masking endpoint already performs BIOES decoding and maps tokens to character offsets.
Scores are model estimates; thresholds need calibration on your intended data.

In [ ]:
scores = call("/scoring", {"model": model, "sequences": [synthetic_email]})
assert len(scores["token_ids"][0]) == len(scores["pruning_scores"][0])
assert all(len(row) == 37 for row in scores["pruning_scores"][0])
print("Scoring alignment passed:", len(scores["token_ids"][0]), "tokens and 37 labels per token.")